# ARCHON Text Mining Workshop

A Very Short Introduction to Text Mining for Archaeologists

**Duration:** 45 minutes

**Objective:** Extract structured information from archaeological texts and link them to a controlled vocabulary. Then compare the output to what an LLM (Large Language Model) can do.

## Workshop Outline
1. How This Works
2. Introduction to Text Mining
3. Setting up the Environment
4. Loading and Processing Text
5. Extracting Archaeological Named Entities
6. Linking Entities to a Controlled Vocabulary


## 1. How This Works

You are currently in Google Colab, an online environment where you can run Python code. Don't worry if you've never programmed before, this workshop can be completed by anyone, regardless of technical level!

IMPORTANT: before we do anything, we have to update a setting in Colab. Click on "Runtime" > "Change Runtime Type", and select "T4 GPU", click "Save". We'll need this GPU power for running a deep learning model later.

You are currently reading instructions, and further down below there are sections of code, called 'code cells'. You can run a code cell by clicking somewhere in the code cell, and pressing SHIFT + ENTER, or clicking the play button to the left of the code. Try this out now in the code cell below. Read the green comments, and update the name, then run the code:



In [ ]:
# This is a so called 'comment' and doesn't do anything, it just literally provides comments on your code

# Comments always start with a hashtag sign, this tells Python to ignore anything else that follows, and Colab makes them green

# Now let's start: replace Indiana Jones with your own name (or a fictional name), be sure to leave the double quotes intact
name = "Indiana Jones"

# Now we'll "print", or display a welcome message:
print(f"Hi {name}, you've just run some Python code!")



Of course, that wasn't super helpful or interesting, but we gotta start somewhere, right?

To give you an idea of some useful stuff we can do with Python, here's an example of making a pie chart. You don't need to understand everything that's going on here, just press play and watch the magic happen:

In [ ]:
# import a 'library' someone else wrote; we're basically re-using their code to do stuff for us. In this case, make a pie chart
import matplotlib.pyplot as plt

# Data for the pie chart
labels = ['Sky', 'Sunny side of pyramid', 'Shady side of pyramid']
sizes = [90, 20, 10]
colors = ['#426391', '#D1A353', '#5A3922']

# Create the pie chart
fig, ax = plt.subplots()
wedges, texts = ax.pie(sizes, colors=colors, startangle=320)

# Add a legend
ax.legend(wedges, labels, title="Legend", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

# Add a title
plt.title("Pie or Pyramid?")

# Show the plot
plt.show()

Ok, you now know how to run Python code in Colab! Time to dive into text mining.

## 2. Introduction to Text Mining
Text mining is the process of deriving high-quality, *structured* information from text (unstructured data). Imagine you have hundreds, or even thousands, of excavation reports from an area and/or time period you are interested in. You might want to extract certain information from this collection, like all C14 dates, all the species of bones, or all the artefact types found at each site. A first step towards this is Named Entity Recognition (NER).

### What is Named Entity Recognition (NER)?
NER is a technique that is used to identify and classify named entities ('things') in text into predefined categories such as places, time periods, person names, organisations, etc. These are the categories often used in generic applications, but of course in archaeology we might also interested in artefacts, contexts, species, etc.

In this workshop, we'll be working with the following 3 example sentences:

1. The excavation at Pompeii revealed a collection of artifacts from the Roman Period, including pottery, coins, and lots of chicken bones.
2. A cremation from the Early Medieval Period was unearthed near Amsterdam, associated with a highly decorated sword.
3. In Göbekli Tepe, a recent dig found multiple stone figurines in a grave context.

For your first assignment, identify and write down all the entities that you think are useful, archaeological information in these sentences. You can write your answers below, and we'll use this as a 'gold standard' to compare to later.

In [ ]:
# Sentence 1:

# Sentence 2:

# Sentence 3:

Well done, you've just done some manual text mining! Now let's see if we can replicate this with various machine learning techniques.

## 3. Setting Up the Environment
We'll use **spaCy**, a popular library for text mining, to do the NER.

First, let's install spaCy and download the English language model.

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

## 4. Loading and Processing Text
Let's load a sample archaeological text and process it with spaCy. This is quite a bit of code, try to read the comments and get an idea of what's happening below:

In [ ]:
# again, we import some code, this time the library created by the spaCy people, and 'pandas' which we use to display tables later on
import spacy
import pandas as pd

# Load the English language model from spaCy
nlp = spacy.load('en_core_web_sm')

# Define our 3 example sentences
text = '''
The excavation at Pompeii revealed a collection of artifacts from the Roman Period, including pottery, coins, and lots of chicken bones.

A cremation from the Early Medieval Period was unearthed near Amsterdam, associated with a highly decorated sword.

In Göbekli Tepe, a recent dig found multiple stone figurines in a grave context.
'''

# Process the text with spaCy
doc = nlp(text)

# Extract named entities from the processed text
entities = [(ent.text, ent.label_) for ent in doc.ents]

# Convert the entities into a table humans can easily read
entities_table = pd.DataFrame(entities, columns=["Entity", "Label"])

# Display the table
entities_table

Ok, you've just done your first text mining with machine learning, and extracted entities from a text! Check the table that is displayed above, does it make sense? Are all the categories correct? Are we missing anything? Reference the Labels to the descriptions below if anything is unclear:

| Label  | Description                                                                        |
|--------|------------------------------------------------------------------------------------|
| LOC    | A physical Location                                                                |
| ORG    | An Organisation                                                                    |
| GPE    | A Geopolitical Entity (everything with a governing body like cities and countries) |
| PERSON | A Person                                                                 |

You probably agree that this result by spaCy is nothing like what you manually extracted.. This is mainly because spaCy is only trained to do generic NER, and extract generic entities, such as names, organisations, locations, etc. It is not created or trained with an archaeological purpose in mind. But luckily, there are alternatives that are specifically built for archaeological NER!





## 5. Extracting Archaeological Named Entities



Now, let's extract named entities from the text that are actually useful for us as archaeologists! We're using a so-called BERT model, that has been pretrained to do archaeological NER.

<div style="float: right; margin: 10px;">
<img src="https://i.ibb.co/99hPLM8w/unnamed.png" style="width:60px;"/>
</div>


Fun fact; BERT is the precursor to the current generation of LLMs like ChatGPT and the like, and uses the same 'Transformer' Deep Learning architecture to learn statistical relations within text.

In the next code cell, we install and load the Python Transformers package, and download 'ArchaeoBERT-NER'. This might take a minute or two as it's a fairly big file (~500MB).

In [ ]:
# install the Transformers package
!pip install -U transformers

# set up a text mining pipeline from Transformers
from transformers import pipeline

# tell the pipeline to load the ArchaeoBERT-NER model
pipe = pipeline("token-classification", model="alexbrandsen/ArchaeoBERT-NER", aggregation_strategy="first")

Now everything is loaded, we can feed our example sentences to ArchaeoBERT:

In [ ]:
# tell the pipeline to process the 3 sentences we defined earlier
entities = pipe(text)

# Convert the output to a table, for ease of reading
ner_table = pd.DataFrame(entities)

# Reorder columns for better readability
ner_table = ner_table[['word', 'entity_group', 'score', 'start', 'end']]

# Display the text and the results table
print(text)
ner_table

Another table to clarify the entity names:

| Label | Description |
|-------|-------------|
| LOC   | Location    |
| ART   | Artefact    |
| SPE   | Species     |
| PER   | Period      |

What do you think of the BERT results? Does this make more sense than the spaCy NER we did earlier?

Now compare these results to the manual NER you did at the start; do you have the same amount of entities, or more/less? Do you have any disagreements with BERT?




## 6. NER with LLMs

Next up, we'll see how well LLMs can do NER.

**Note**: if you don't want to use LLMs for any reason, you can skip this section.

1. Open your favourite LLM (if you don't have one, [Gemini](https://gemini.google.com/app) is easy as you already have a Google account)
2. Write a prompt that asks the LLM to do NER on our 3 example sentences
3. Try and steer the LLM to only predict the 4 entity types that BERT predicted
4. Try and get the output as a table, so it's easy to compare
5. Compare the output to the BERT output, and your manual text mining
6. Also compare the speed of the process, how long does the LLM take compared to running that cell above where we run BERT (re-run it to check!). Do you think this is scalable and affordable when processing large amounts of documents?



## 7. Linking Entities to a Controlled Vocabulary
A controlled vocabulary is a predefined list of terms. The most well known one is called wikidata.org, and is used extensively by all sorts of projects, including wikipedia. In the cells below, we're doing some more advanced stuff, so don't worry about understanding the code.

We're going to first look up the locations we found in our text in wikidata, and then we'll retrieve the location (in latitude/longitude), the inception year of the city, and an image of the city. This way, through the power of Linked Open Data, we can add more information to the information we extracted from our own text:

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import re
from IPython.display import display

# Filter locations from the ner_table
locations_raw = ner_table[ner_table['entity_group'] == 'LOC']['word'].unique()

# Prepare a list to store results
wikidata_results = []

print(f"Attempting to retrieve Wikidata info for {len(locations_raw)} unique locations using SPARQL...")

# Initialize SPARQL endpoint
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)

for loc_name in locations_raw:
    qid = None
    latitude = None
    longitude = None
    inception = None
    image_link = None
    processed_loc_name = loc_name.strip()

    try:
        # SPARQL query to get QID, P625 (geographic coordinates), P571 (inception), and P18 (image) for a given label
        sparql_query = f"""
        SELECT ?item ?itemLabel ?coordinates ?inception ?image WHERE {{
          ?item rdfs:label "{processed_loc_name}"@en .
          OPTIONAL {{ ?item wdt:P625 ?coordinates . }}
          OPTIONAL {{ ?item wdt:P571 ?inception . }}
          OPTIONAL {{ ?item wdt:P18 ?image . }}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        LIMIT 1
        """
        sparql.setQuery(sparql_query)
        results = sparql.query().convert()

        if results and results['results']['bindings']:
            binding = results['results']['bindings'][0]
            if 'item' in binding:
                # Extract QID from the URI (e.g., http://www.wikidata.org/entity/Q43332)
                qid = binding['item']['value'].split('/')[-1]

            if 'coordinates' in binding:
                coord_str = binding['coordinates']['value']
                # P625 values are typically in 'Point(longitude latitude)' format
                # Corrected regex: use single backslash to escape regex special characters
                match = re.search(r'Point\(([-+]?\d*\.?\d+)\s+([-+]?\d*\.?\d+)\)', coord_str)
                if match:
                    longitude = float(match.group(1))
                    latitude = float(match.group(2))

            if 'inception' in binding:
                inception = binding['inception']['value']

            if 'image' in binding:
                image_link = binding['image']['value']

    except Exception as e:
        print(f"Error processing {processed_loc_name} with SPARQL: {e}")
        # Continue to the next location even if an error occurs

    wikidata_results.append({
        'Location': processed_loc_name,
        'Wikidata QID': qid,
        'Latitude': latitude,
        'Longitude': longitude,
        'Inception': inception,
        'Image Link': image_link
    })

# Convert results to a DataFrame
wikidata_df = pd.DataFrame(wikidata_results)

# Display the DataFrame
print("\nWikidata Location Linking Results (SPARQL):")
display(wikidata_df)


Great, now we have all the information! We can now make an interactive map showing all the locations we found in our text, and also show new information; like the inception year of the city, and when you click on the city, we'll display an image of the city:

In [ ]:
import folium
import pandas as pd # Ensure pandas is imported for pd.notna
import re # Import the re module for regular expressions

# Filter out rows where Latitude or Longitude are NaN (e.g., for non-location entities)
locations_for_map = wikidata_df.dropna(subset=['Latitude', 'Longitude'])

# Calculate the center of the map based on the average latitude and longitude
center_lat = locations_for_map['Latitude'].mean()
center_lon = locations_for_map['Longitude'].mean()

# Create a Folium map object using 'CartoDB Positron' tiles
m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles='CartoDB Positron')

# Add a marker for each location
for index, row in locations_for_map.iterrows():
    # Format inception date for tooltip
    inception_year = "N/A"
    if isinstance(row['Inception'], str) and row['Inception']:
        # Use regex to extract the year, handling potential leading minus sign
        match = re.match(r'^(-?\d+)', row['Inception'])
        if match:
            inception_year = match.group(1)

    # Create tooltip content
    tooltip_text = f"{row['Location']}"
    if inception_year != "N/A":
        tooltip_text += f" (Inception: {inception_year})"

    # Create popup HTML content with image
    popup_html = f"<h3>{row['Location']}</h3>"
    if isinstance(row['Image Link'], str) and row['Image Link']:
        popup_html += f"<img src='{row['Image Link']}' width='200px'>"
    else:
        popup_html += "<p>No image available</p>"

    # Create the popup object
    popup_object = folium.Popup(popup_html, max_width=300) # Set max_width for better display

    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=popup_object, # Use the Folium Popup object
        tooltip=tooltip_text # Use the formatted tooltip text
    ).add_to(m)

# Display the map
display(m)

This is of course just a toy example, but shows you how you can relatively easily get structured data from text, combine it with other Linked Open Data, and display it together. Of course you can also export the information as a spreadsheet for manual inspection and/or further analysis.

## 8. Conclusions

You've now had some experience with text mining, but of course this is just the tip of the ice berg in terms of what we can do with these kinds of techniques. If you want to learn more, I have a couple of other workshops and courses available online you can self-study with:

* [Introduction to Programming for Archaeologists](https://github.com/alexbrandsen/Introduction-to-Programming-for-Archaeologists), which goes from the basics of programming in Python, all the way to training your own machine learning model
* [Making Graphs in Python for Archaeologists](https://github.com/alexbrandsen/making-graphs-in-python-for-archaeologists), a short workshop (~60 mins) on how to make various graphs in Python
* [Machine Learning Seminar](https://github.com/alexbrandsen/TIFA-seminar-ICArEHB), a more in-depth seminar in multiple parts, dealing with machine learning on tabular data, images and text